In [5]:
!pip install pingouin

In [6]:
## 1. Nạp và tiền xử lý dữ liệu
#- Đọc tập dữ liệu `data_cleaned (2).csv` chứa vector chênh lệch của 4 chỉ số KPI.
#- Xử lý kiểu dữ liệu (nếu bị dính chuỗi do Excel xuất ra).
import pandas as pd
import numpy as np
from scipy.stats import chi2, f
import pingouin as pg

# Đọc data từ file
df = pd.read_csv('data_cleaned_n60.csv')

# Khai báo 4 cột chênh lệch (D = Trong - Trước)
cols = ['d_DoanhSo', 'd_LuongKhach', 'd_SKU', 'd_TyLeChuyenDoi']

print(df[cols].dtypes)  # THÊM: xem cột nào đang là object (chuỗi)

# Clean sơ lại data đề phòng lỗi định dạng string có dấu chấm/phẩy
# (chỉ đúng khi số viết kiểu VN như 1.234,56; nếu cột đã là float64 thì không chạy vào đây)
for c in cols:
    if df[c].dtype == 'object':
        df[c] = df[c].astype(str).str.replace('.', '', regex=False).str.replace(',', '.', regex=False).astype(float)

# THÊM: bỏ dòng thiếu để ma trận S không bị lỗi
print("Số ô thiếu mỗi cột:")
print(df[cols].isna().sum())
df = df.dropna(subset=cols).reset_index(drop=True)

D = df[cols].values
n, p = D.shape

print(f"Cỡ mẫu n = {n}")
print(f"Số biến p = {p}")

# Tính các ma trận cơ bản cho mẫu
d_bar = np.mean(D, axis=0)
S = np.cov(D, rowvar=False)
S_inv = np.linalg.pinv(S)

## 2. Tiền xử lý và Đặc trưng mẫu (Mục 3.1 & 3.2)
#- Lọc Outlier bằng khoảng cách Mahalanobis (ngưỡng alpha = 0.001).
#- Tính định thức ma trận hiệp phương sai |S| và ma trận tương quan R.
# --- 3.1 Tìm Outlier bằng Mahalanobis ---
def calc_mahalanobis(x, mean_v, inv_cov):
    diff = x - mean_v
    return diff @ inv_cov @ diff.T

df['Mahalanobis'] = [calc_mahalanobis(row, d_bar, S_inv) for row in D]
chi2_thresh = chi2.ppf(0.999, df=p)
outliers = df[df['Mahalanobis'] > chi2_thresh]

print("--- 3.1. KHOẢNG CÁCH MAHALANOBIS ---")
print(f"Ngưỡng chi2 tới hạn: {chi2_thresh:.4f}")
print(f"Số lượng outlier tìm thấy: {len(outliers)}")

# SỬA: loại outlier thật rồi tính lại mọi thứ trên data sạch
df = df[df['Mahalanobis'] <= chi2_thresh].reset_index(drop=True)
D = df[cols].values
n = D.shape[0]
d_bar = np.mean(D, axis=0)
S = np.cov(D, rowvar=False)
S_inv = np.linalg.pinv(S)

print(f"Cỡ mẫu sau khi loại outlier: n = {n}")
if n <= 50:
    print("Cảnh báo: n <= 50, không đủ cỡ mẫu tối thiểu!")

# --- 3.2 Đặc trưng mẫu đa biến ---
det_S = np.linalg.det(S)
R = np.corrcoef(D, rowvar=False)

print("\n--- 3.2. ĐẶC TRƯNG MẪU ---")
# THÊM: in vector trung bình và ma trận S
print("Vector trung bình chênh lệch:")
for name, val in zip(cols, d_bar):
    print(f"  {name}: {val:.4f}")
print("\nMa trận hiệp phương sai S:")
print(pd.DataFrame(S, index=cols, columns=cols).round(4))
print(f"\nĐịnh thức |S| = {det_S:.4e}")
print("\nMa trận tương quan R:")
print(pd.DataFrame(R, index=cols, columns=cols).round(4))
# THÊM: kiểm tra S có khả nghịch tốt không
print(f"\nHạng của S = {np.linalg.matrix_rank(S)} (cần = {p})")
print(f"Số điều kiện của R = {np.linalg.cond(R):.2f}")

## 3. Kiểm tra giả định phân phối chuẩn đa biến (Mục 3.3)
#- Sử dụng kiểm định Henze-Zirkler (HZ) từ thư viện pingouin.
hz_test = pg.multivariate_normality(df[cols], alpha=0.05)

print("\n--- 3.3. KIỂM ĐỊNH CHUẨN ĐA BIẾN (HZ) ---")
print(f"Thống kê HZ: {hz_test.hz:.4f}")
print(f"p-value: {hz_test.pval:.6e}")
print(f"Kết luận chuẩn đa biến?: {hz_test.normal}")

## 4. Thực thi kiểm định Hotelling's T² (Mục 3.4)
#- Tính thống kê T² và chuyển đổi sang phân phối F.
#- Đưa ra kết luận thống kê.
# Tính thống kê Hotelling T2
T2 = n * (d_bar.T @ S_inv @ d_bar)

# Đổi sang F
df1 = p
df2 = n - p
F_stat = ((n - p) / (p * (n - 1))) * T2

# Tính p-value và F tới hạn
alpha = 0.05
F_crit = f.ppf(1 - alpha, df1, df2)
p_val = f.sf(F_stat, df1, df2)

print("\n--- 3.4. KẾT QUẢ HOTELLING T2 ---")
print("Vector trung bình d_bar:")
for name, val in zip(cols, d_bar):
    print(f"  {name}: {val:.4f}")

print(f"\nT2 = {T2:.4f}")
print(f"F = {F_stat:.4f}")
print(f"df1 = {df1}, df2 = {df2}")
print(f"F tới hạn = {F_crit:.4f}")
print(f"p-value = {p_val:.6e}")
print(f"Mức ý nghĩa alpha = {alpha}")

# THÊM: tính lại T2 sau khi chuẩn hóa thang đo để kiểm tra pinv không làm sai số
Ds = D / D.std(axis=0, ddof=1)
ds_bar = Ds.mean(axis=0)
T2_check = n * (ds_bar @ np.linalg.solve(np.cov(Ds, rowvar=False), ds_bar))
print(f"T2 kiểm tra (đã chuẩn hóa) = {T2_check:.4f}  (phải xấp xỉ T2 ở trên)")

print("\n--- KẾT LUẬN ---")
if p_val < alpha:
    print("BÁC BỎ H0.")
    print("Có sự khác biệt có ý nghĩa thống kê về 4 chỉ số KPI giữa 2 giai đoạn.")
else:
    print("KHÔNG BÁC BỎ H0.")
    print("Chưa đủ bằng chứng để kết luận có sự khác biệt giữa 2 giai đoạn.")

d_DoanhSo          float64
d_LuongKhach       float64
d_SKU              float64
d_TyLeChuyenDoi    float64
dtype: object
Số ô thiếu mỗi cột:
d_DoanhSo          0
d_LuongKhach       0
d_SKU              0
d_TyLeChuyenDoi    0
dtype: int64
Cỡ mẫu n = 60
Số biến p = 4
--- 3.1. KHOẢNG CÁCH MAHALANOBIS ---
Ngưỡng chi2 tới hạn: 18.4668
Số lượng outlier tìm thấy: 0
Cỡ mẫu sau khi loại outlier: n = 60

--- 3.2. ĐẶC TRƯNG MẪU ---
Vector trung bình chênh lệch:
  d_DoanhSo: 144737.3154
  d_LuongKhach: 3.3958
  d_SKU: 0.6000
  d_TyLeChuyenDoi: 0.0436

Ma trận hiệp phương sai S:
                    d_DoanhSo  d_LuongKhach       d_SKU  d_TyLeChuyenDoi
d_DoanhSo        7.803413e+09    67788.5091  11466.7487         797.6853
d_LuongKhach     6.778851e+04        8.8375      0.2966           0.1284
d_SKU            1.146675e+04        0.2966      0.6148           0.0003
d_TyLeChuyenDoi  7.976853e+02        0.1284      0.0003           0.0019

Định thức |S| = 5.5740e+05

Ma trận tương quan R:
          